# 🚀 Advanced Football Feature Engineering

This notebook focuses on the development of advanced analytical features designed to improve football match prediction performance and extract deeper competitive insights from historical international football data.

The engineered features aim to capture momentum, consistency, contextual strength, temporal behavior, and advanced team performance dynamics.

---

## 📌 Main Objectives

- Create advanced predictive features
- Build contextual and temporal indicators
- Develop rolling and weighted statistics
- Capture team momentum and consistency
- Enhance model interpretability and predictive performance

---


In [ ]:
# 🎨 Visualization Style Configuration

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

sns.set_theme(style="whitegrid", palette="viridis")


# ============================================
# 🌍 FIFA World Cup Project
# 🚀 Advanced Feature Engineering
# ============================================

---

In [ ]:
# ============================================
# 📚 Import Libraries
# ============================================

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [ ]:
# ============================================
# 📥 Load Feature Dataset
# ============================================

results = pd.read_csv(
    "../data/processed/results_features.csv"
)

results["date"] = pd.to_datetime(
    results["date"]
)

results = (
    results
    .sort_values("date")
    .reset_index(drop=True)
)

results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,match_result,goal_difference,total_goals,home_advantage,tournament_weight,time_weight,combined_weight
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1872,Draw,0.0,0.0,1,1.0,-0.54,-0.54
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1873,Home Win,2.0,6.0,1,1.0,-0.53,-0.53
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1874,Home Win,1.0,3.0,1,1.0,-0.52,-0.52
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1875,Draw,0.0,4.0,1,1.0,-0.51,-0.51
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1876,Home Win,3.0,3.0,1,1.0,-0.50,-0.50


In [ ]:
# 🇩🇪 Data Standardization: Germany Unification

# Automatically detect dataframe variables
possible_dfs = [
    "df",
    "matches",
    "results",
    "world_cup",
    "team_metrics",
    "data"
]

for df_name in possible_dfs:

    if df_name in globals():

        dataframe = globals()[df_name]

        country_columns = [
            col for col in dataframe.columns
            if any(keyword in col.lower()
                   for keyword in ["team", "country", "winner", "home", "away"])
        ]

        for col in country_columns:
            dataframe[col] = dataframe[col].replace("West Germany", "Germany")

        print(f"✅ Germany standardization applied to: {df_name}")


## 🏟️ Contextual Match Features

Analyzing home advantage, away performance, and neutral venue effects.

## 📅 Temporal Features

Extracting historical and chronological behavioral patterns.

In [ ]:
# ============================================
# 🔄 Team Perspective Dataset
# ============================================

home_df = pd.DataFrame({

    "date": results["date"],
    "team": results["home_team"],
    "opponent": results["away_team"],

    "goals_scored": results["home_score"],
    "goals_conceded": results["away_score"],

    "goal_difference": (
        results["home_score"] -
        results["away_score"]
    ),

    "is_home": 1,

    "match_points": np.where(
        results["home_score"] >
        results["away_score"],

        3,

        np.where(
            results["home_score"] ==
            results["away_score"],
            1,
            0
        )
    )
})

In [ ]:
away_df = pd.DataFrame({

    "date": results["date"],
    "team": results["away_team"],
    "opponent": results["home_team"],

    "goals_scored": results["away_score"],
    "goals_conceded": results["home_score"],

    "goal_difference": (
        results["away_score"] -
        results["home_score"]
    ),

    "is_home": 0,

    "match_points": np.where(
        results["away_score"] >
        results["home_score"],

        3,

        np.where(
            results["away_score"] ==
            results["home_score"],
            1,
            0
        )
    )
})

## 🗂️ Dataset Overview

Loading and validating datasets used for advanced feature generation.

In [ ]:
team_df = pd.concat(
    [home_df, away_df],
    ignore_index=True
)

team_df = (
    team_df
    .sort_values("date")
)

team_df.head()

,date,team,opponent,goals_scored,goals_conceded,goal_difference,is_home,match_points
0,1872-11-30,Scotland,England,0.0,0.0,0.0,1,1
48589,1872-11-30,England,Scotland,0.0,0.0,0.0,0,1
48590,1873-03-08,Scotland,England,2.0,4.0,-2.0,0,0
1,1873-03-08,England,Scotland,4.0,2.0,2.0,1,3
2,1874-03-07,Scotland,England,2.0,1.0,1.0,1,3


## 🔄 Rolling Metrics

Generating moving averages and dynamic performance indicators.

## 📈 Momentum & Team Form Analysis

Capturing consistency, streaks, and recent competitive behavior.

In [ ]:
# ============================================
# 📈 Rolling Average Points
# ============================================

team_df["avg_points_last_5"] = (

    team_df
    .groupby("team")["match_points"]

    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .mean()
    )
)

In [ ]:
# ============================================
# ⚽ Rolling Goals Scored
# ============================================

team_df["avg_goals_scored_last_5"] = (

    team_df
    .groupby("team")["goals_scored"]

    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .mean()
    )
)

In [ ]:
# ============================================
# 🛡️ Rolling Goals Conceded
# ============================================

team_df["avg_goals_conceded_last_5"] = (

    team_df
    .groupby("team")["goals_conceded"]

    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .mean()
    )
)

In [ ]:
# ============================================
# 🔥 Rolling Goal Difference
# ============================================

team_df["avg_goal_diff_last_5"] = (

    team_df
    .groupby("team")["goal_difference"]

    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .mean()
    )
)

In [ ]:
# ============================================
# 🔥 Win Streak
# ============================================

team_df["win"] = (
    team_df["match_points"] == 3
).astype(int)

In [ ]:
team_df["win_streak"] = (

    team_df
    .groupby("team")["win"]

    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .sum()
    )
)

In [ ]:
# ============================================
# 🧤 Clean Sheets
# ============================================

team_df["clean_sheet"] = (
    team_df["goals_conceded"] == 0
).astype(int)

team_df["clean_sheets_last_5"] = (

    team_df
    .groupby("team")["clean_sheet"]

    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .sum()
    )
)

In [ ]:
# ============================================
# 🚀 Team Momentum Score
# ============================================

team_df["momentum_score"] = (

    team_df["avg_points_last_5"] * 0.4 +

    team_df["avg_goal_diff_last_5"] * 0.3 +

    team_df["win_streak"] * 0.2 +

    team_df["clean_sheets_last_5"] * 0.1
)

## 🚀 Advanced Feature Engineering

Creating high-level analytical variables and predictive indicators.

In [ ]:
# ============================================
# 👀 Feature Preview
# ============================================

team_df.head(10)

,date,team,opponent,goals_scored,goals_conceded,goal_difference,is_home,match_points,avg_points_last_5,avg_goals_scored_last_5,avg_goals_conceded_last_5,avg_goal_diff_last_5,win,win_streak,clean_sheet,clean_sheets_last_5,momentum_score
0,1872-11-30,Scotland,England,0.0,0.0,0.0,1,1,NaN,NaN,NaN,NaN,0,NaN,1,NaN,NaN
48589,1872-11-30,England,Scotland,0.0,0.0,0.0,0,1,NaN,NaN,NaN,NaN,0,NaN,1,NaN,NaN
48590,1873-03-08,Scotland,England,2.0,4.0,-2.0,0,0,1.000000,0.000000,0.000000,0.000000,0,0.0,0,1.0,5.000000e-01
1,1873-03-08,England,Scotland,4.0,2.0,2.0,1,3,1.000000,0.000000,0.000000,0.000000,1,0.0,0,1.0,5.000000e-01
2,1874-03-07,Scotland,England,2.0,1.0,1.0,1,3,0.500000,1.000000,2.000000,-1.000000,1,0.0,0,1.0,2.775558e-17
48591,1874-03-07,England,Scotland,1.0,2.0,-1.0,0,0,2.000000,2.000000,1.000000,1.000000,0,1.0,0,1.0,1.400000e+00
3,1875-03-06,England,Scotland,2.0,2.0,0.0,1,1,1.333333,1.666667,1.333333,0.333333,0,1.0,0,1.0,9.333333e-01
48592,1875-03-06,Scotland,England,2.0,2.0,0.0,0,1,1.333333,1.333333,1.666667,-0.333333,0,1.0,0,1.0,7.333333e-01
4,1876-03-04,Scotland,England,3.0,0.0,3.0,1,3,1.250000,1.500000,1.750000,-0.250000,1,1.0,1,1.0,7.250000e-01
48593,1876-03-04,England,Scotland,0.0,3.0,-3.0,0,0,1.250000,1.750000,1.500000,0.250000,0,1.0,0,1.0,8.750000e-01


In [ ]:
# ============================================
# 💾 Save Advanced Features Dataset
# ============================================

team_df.to_csv(
    "../data/processed/advanced_features.csv",
    index=False
)

print(
    "✅ Advanced feature dataset saved successfully."
)

✅ Advanced feature dataset saved successfully.


💡 📌 Observation:

The advanced feature engineering process introduced rolling performance metrics, momentum indicators, and dynamic team statistics that better capture recent football performance trends.

Unlike static historical metrics, these rolling features allow the future predictive models to incorporate team form, consistency, offensive efficiency, and defensive stability over time.

---

🏁 🎯 Conclusion:

The dataset now contains contextual and temporal football intelligence features that simulate how analysts evaluate real-world team performance.

These advanced metrics significantly improve the predictive capability of future machine learning models by incorporating momentum, recent form, and match consistency into the analytical pipeline.

---